AIM: house price prediction using deep feed forward neural network or Kaggle housing dataset.
activities to perform: dataset - data preprocessing snd normalization and designing deep feed forward neural network architecture.
for prediction rate evaluate mse rmsc mle . explore diff no of layers to improve the performance of the architecture.

In [ ]:
# ============================================================
# HOUSE PRICE PREDICTION — DEEP FEED-FORWARD NEURAL NETWORK
# Dataset: Kaggle "House Prices" (via kagglehub mirror)
# ============================================================

!pip install -q kagglehub

import kagglehub
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ------------------------------------------------------------
# 0. DOWNLOAD DATASET
# ------------------------------------------------------------
path = kagglehub.dataset_download("lespin/house-prices-dataset")
print("Path to dataset files:", path)
print("Files found:", os.listdir(path))

# NOTE: verify this picks the TRAIN file, not a test/sample file,
# if the folder has more than one CSV
csv_file = "train.csv" # Changed to explicitly select train.csv
df = pd.read_csv(os.path.join(path, csv_file))
print("Shape:", df.shape)

# ------------------------------------------------------------
# 1. PREPROCESSING
# ------------------------------------------------------------
target = 'SalePrice'
df = df.drop(columns=['Id'], errors='ignore')

y = df[target]
X = df.drop(columns=[target])

# Log-transform target — SalePrice is right-skewed; skipping this
# distorts MSE/RMSE and makes the layer comparison meaningless
y_log = np.log1p(y)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

for c in num_cols:
    X[c] = X[c].fillna(X[c].median())
for c in cat_cols:
    X[c] = X[c].fillna('Missing')

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_temp, y_train, y_temp = train_test_split(X, y_log, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Fit scaler ONLY on train — fitting on all data leaks test info
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

input_dim = X_train_s.shape[1]
print("Input features after encoding:", input_dim)

# ------------------------------------------------------------
# 2. MODEL BUILDER (DFNN, configurable depth)
# ------------------------------------------------------------
def build_model(n_hidden_layers=2, units=128, dropout=0.2, lr=1e-3):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for i in range(n_hidden_layers):
        u = max(units // (2 ** i), 16)
        model.add(layers.Dense(u, activation='relu'))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='linear'))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss='mse', metrics=['mae'])
    return model

def evaluate(model, X_test_s, y_test):
    preds_log = model.predict(X_test_s, verbose=0).flatten()
    preds = np.expm1(preds_log)
    actual = np.expm1(y_test)
    mse = mean_squared_error(actual, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual, preds)
    return mse, rmse, mae

# ------------------------------------------------------------
# 3. BASELINE MODEL
# ------------------------------------------------------------
baseline = build_model(n_hidden_layers=2)
baseline.fit(X_train_s, y_train, validation_data=(X_val_s, y_val),
             epochs=100, batch_size=32, verbose=0,
             callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])

mse, rmse, mae = evaluate(baseline, X_test_s, y_test)
print(f"\nBaseline (2 hidden layers) -> MSE: {mse:,.0f}  RMSE: {rmse:,.0f}  MAE: {mae:,.0f}")

# ------------------------------------------------------------
# 4. EXPLORE DIFFERENT NUMBERS OF LAYERS
# ------------------------------------------------------------
results = []
for n_layers in [1, 2, 3, 4, 5]:
    model = build_model(n_hidden_layers=n_layers)
    model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val),
              epochs=100, batch_size=32, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    mse, rmse, mae = evaluate(model, X_test_s, y_test)
    results.append({'hidden_layers': n_layers, 'MSE': mse, 'RMSE': rmse, 'MAE': mae})
    print(f"Layers: {n_layers} -> MSE: {mse:,.0f}  RMSE: {rmse:,.0f}  MAE: {mae:,.0f}")

results_df = pd.DataFrame(results)
print("\n=== Layer depth comparison ===")
print(results_df.to_string(index=False))

Using Colab cache for faster access to the 'house-prices-dataset' dataset.
Path to dataset files: /kaggle/input/house-prices-dataset
Files found: ['sample_submission.csv', 'data_description.txt', 'train.csv', 'test.csv']
Shape: (1460, 81)
Input features after encoding: 260

Baseline (2 hidden layers) -> MSE: 3,829,151,262,716,322,304  RMSE: 1,956,821,725  MAE: 132,361,913
Layers: 1 -> MSE: 104,952,172,810,870  RMSE: 10,244,617  MAE: 776,996


Layers: 2 -> MSE: 220,082,821,744,714,752  RMSE: 469,129,856  MAE: 31,779,581


Layers: 3 -> MSE: 118,952,115,979,850  RMSE: 10,906,517  MAE: 1,163,385
Layers: 4 -> MSE: 48,509,564,629,903,264  RMSE: 220,248,870  MAE: 15,089,430
Layers: 5 -> MSE: 692,810,944,588,486  RMSE: 26,321,302  MAE: 1,930,602

=== Layer depth comparison ===
 hidden_layers          MSE         RMSE          MAE
             1 1.049522e+14 1.024462e+07 7.769965e+05
             2 2.200828e+17 4.691299e+08 3.177958e+07
             3 1.189521e+14 1.090652e+07 1.163385e+06
             4 4.850956e+16 2.202489e+08 1.508943e+07
             5 6.928109e+14 2.632130e+07 1.930602e+06
